# PATSTAT EP Register - exercise 1
In this exercise we will gradually build a query using ORM that gives us all the Swiss applicants related to European patents granted after the 31 December 2019. 

## The publications table
First we need to learn about the table `reg102_pat_publn`. This table contains details about European and international publications that are visible in the European Patent Register. 


### EP publications from 2020 onwards
We will use the filter functionality to see only EP publications published since 2020, with these filters. 

- `publn_auth == 'EP'`: Only includes records where the publication authority is 'EP'.
- `publn_date > '2019-12-31'`: Only includes records where the publication date is after 31 December 2019.
- `publn_date < '2099-12-31'`: PATSTAT uses the date 2099-12-31 instead of `null` to indicate that there is no data information for a specific record. With this condition we filter out publications with an unknown date.

The results are ordered by the `publn_date` column in ascending order.


In [1]:
# Importing the PATSTAT client
from epo.tipdata.patstat import PatstatClient

# Initialise the PATSTAT client
patstat = PatstatClient()

# Access ORM
db = patstat.orm()

# Importing tables as models
from epo.tipdata.patstat.database.models import REG102_PAT_PUBLN



This client instance is currently configured to use a test dataset with reduced number of publications (~10K).
Use PatstatClient(env='PROD') to use the complete PATSTAT dataset (>140M publications).
Use PatstatClient(env='TEST') to use the test dataset and avoid displaying this warning



In [2]:
q = db.query(
    REG102_PAT_PUBLN.publn_auth,
    REG102_PAT_PUBLN.publn_nr,
    REG102_PAT_PUBLN.publn_date
).filter(
    REG102_PAT_PUBLN.publn_kind == 'B1', # shows only granted patents
    REG102_PAT_PUBLN.publn_auth == 'EP',
    REG102_PAT_PUBLN.publn_date > '2019-12-31',
    REG102_PAT_PUBLN.publn_date < '9999-12-31' # eliminates publications without a date
).order_by(
    REG102_PAT_PUBLN.publn_date
)
# Creating a dataframe with the results
res = patstat.df(q)

res


,publn_auth,publn_nr,publn_date
0,EP,3454159,2020-01-01
1,EP,3269997,2020-01-01
2,EP,2202862,2020-01-01
3,EP,3153398,2020-01-01
4,EP,3399297,2020-01-01
...,...,...,...
2998,EP,4051901,2024-03-13
2999,EP,4191058,2024-03-13
3000,EP,3704374,2024-03-13
3001,EP,3670899,2024-03-13


## Introducing the reg107_parties table
The goal of this exercise is to find out the European patents that mention an applicant or proprietor from Switzerland. For this we need to work with the `reg107_parties` table in the PATSTAT EP Register database. This table stores information about the parties involved in patent applications. 



### Finding Swiss applicants
We will make a simple query in the `reg107_parties` to find all the parties with a place of residence in Switzerland. There can be three types of parties:

* **Applicant or proprietor ("A")**
* **Inventor ("I")**
* **Agent or representative ("R")**
* **Opponent ("O")**

We will query the table 107 with these filters:


- `REG107_PARTIES.country == 'CH'`: Only includes records that specify a place of business or residence in Switzerland.
- `REG107_PARTIES.type == 'A'`: Only includes records listed as applicant or proprietor.
- `REG107_PARTIES.is_latest == 'Y'`: the parties for an application change over time. This field is a Y/N flag. 'Y' indicates that the record belongs to the latest (current or most recent) set of applicants, inventors, representatives, or opponents. 

In [3]:
from epo.tipdata.patstat.database.models import REG107_PARTIES

q = db.query(
    REG107_PARTIES.name,
    ).filter(
    REG107_PARTIES.country == 'CH',
    REG107_PARTIES.type == 'A',
    REG107_PARTIES.is_latest == 'Y'
).order_by(
    REG107_PARTIES.name
)

# Creating a dataframe with the results
res = patstat.df(q)

res[0:10]


,name
0,ABB Power Grids Switzerland AG
1,ABB RESEARCH LTD.
2,ABB Research Ltd
3,ABB Research Ltd.
4,ABB Research Ltd.
5,ABB Research Ltd.
6,ABB Research Ltd.
7,ABB Research Ltd.
8,ABB Research Ltd.
9,ABB Research Ltd.


### Understanding duplicates in the parties table
Unfortunately, there is no unique identifier for applicants, inventors, or proprietors in the field of patent data. Each patent application mentions the applicants with their names and addresses. It often happens that the same applicant is filed with variations of the same address, or different addresses, in different patent applications. The applicant name itself can sometimes be spelled differently. This typically creates multiple records in the parties table for one single legal entity or a single person. 

Please take into consideration this fact for all your patent data analysis.

## Joining parties to publications via the applications table

If you look at the logical model diagram of PATSTAT in the documentation, you will see that the `reg107_parties` and `reg102_pat_publication` tables are not related. In PATSTAT EP Register the central table is `reg101_appln`, which contains data about the European and international patent applications in the Register. 

We will then join table 102, 101, and 107 to get the desired query. 

Let's first join tables 101 and 102 to get the publications from 2020 and later, and get the application ID for each publication. This application ID will later be needed for joining tables 101 and 107.  

In [4]:
from epo.tipdata.patstat.database.models import REG101_APPLN

q = db.query(
    REG102_PAT_PUBLN.publn_auth,
    REG102_PAT_PUBLN.publn_nr,
    REG102_PAT_PUBLN.publn_date,
    REG101_APPLN.id,
    REG101_APPLN.appln_nr
).join(
    REG101_APPLN, REG102_PAT_PUBLN.id == REG101_APPLN.id
).filter(
    REG102_PAT_PUBLN.publn_kind == 'B1',
    REG102_PAT_PUBLN.publn_auth == 'EP',
    REG102_PAT_PUBLN.publn_date > '2019-12-31',
    REG102_PAT_PUBLN.publn_date < '9999-12-31'
).order_by(
    REG102_PAT_PUBLN.publn_date
)

# Creating a dataframe with the results
res = patstat.df(q)
res


,publn_auth,publn_nr,publn_date,id,appln_nr
0,EP,3454159,2020-01-01,17382598,17382598
1,EP,3269997,2020-01-01,16179462,16179462
2,EP,2202862,2020-01-01,9014851,9014851
3,EP,3153398,2020-01-01,15800536,15800536
4,EP,3399297,2020-01-01,16880789,16880789
...,...,...,...,...,...
2998,EP,4051901,2024-03-13,19808695,19808695
2999,EP,4191058,2024-03-13,23152372,23152372
3000,EP,3704374,2024-03-13,18807560,18807560
3001,EP,3670899,2024-03-13,18382969,18382969


### Our final query

We are reaching the end of the exercise. We are ready now to build a query that connects the 101, 102 and 107 tables, and looks for Swiss applicants related to European patents granted from 2020 and onwards. 

The query performs a double join to connect three tables: `REG101_APPLN`, `REG102_PAT_PUBLN`, and `REG107_PARTIES`.

1. **First Join**:

   - Connects `REG101_APPLN` and `REG102_PAT_PUBLN` using `REG102_PAT_PUBLN.id == REG101_APPLN.id`.

2. **Second Join**:
   - Connects the resulting dataset with `REG107_PARTIES` using `REG101_APPLN.id == REG107_PARTIES.id`.



In [5]:
q = db.query(
    REG102_PAT_PUBLN.publn_auth,
    REG102_PAT_PUBLN.publn_nr,
    REG102_PAT_PUBLN.publn_date,
    REG101_APPLN.id,
    REG101_APPLN.appln_nr,
    REG107_PARTIES.name
).join(
    REG101_APPLN, REG102_PAT_PUBLN.id == REG101_APPLN.id
).join(
    REG107_PARTIES, REG101_APPLN.id == REG107_PARTIES.id
).filter(
    REG102_PAT_PUBLN.publn_kind == 'B1',
    REG102_PAT_PUBLN.publn_auth == 'EP',
    REG102_PAT_PUBLN.publn_date > '2019-12-31',
    REG102_PAT_PUBLN.publn_date < '9999-12-31',
    REG107_PARTIES.country == 'CH',
    REG107_PARTIES.type == 'A',
    REG107_PARTIES.is_latest == 'Y'
).order_by(
    REG107_PARTIES.name
)

# Creating a dataframe with the results
res = patstat.df(q)
res


,publn_auth,publn_nr,publn_date,id,appln_nr,name
0,EP,3590690,2020-12-23,18181594,18181594,ABB Power Grids Switzerland AG
1,EP,3610341,2023-08-30,18711816,18711816,ABB Schweiz AG
2,EP,3298273,2020-03-04,16723140,16723140,ABB Schweiz AG
3,EP,3465047,2020-12-16,17726620,17726620,ABB Schweiz AG
4,EP,3411594,2021-03-31,17702397,17702397,ABB Schweiz AG
5,EP,3595167,2021-12-01,18182814,18182814,ABB Schweiz AG
6,EP,3534522,2021-04-28,18159233,18159233,ABB Schweiz AG
7,EP,3613137,2022-10-19,18717398,18717398,ABB Schweiz AG
8,EP,3411758,2022-05-18,17748348,17748348,ABB Schweiz AG
9,EP,2871090,2020-10-07,13191822,13191822,ABB Schweiz AG
